# End-to-End Scikit-HEP Analysis Pipeline

This chapter brings together everything you learned—`uproot`, `awkward`, `vector`, and `hist`—into a realistic, end-to-end physics analysis pipeline. Our goal is to "rediscover" the Higgs boson in the $H \rightarrow ZZ^* \rightarrow 4\ell$ channel using ATLAS Open Data.

We will use a dataset of proton-proton collisions. The notebook processes the data in chunks, meaning it will run with a small, fixed memory footprint no matter how large the dataset is.

In [ ]:
import awkward as ak
import hist
import matplotlib.pyplot as plt
import uproot
import vector
from hepunits import GeV, MeV
from particle import Particle

vector.register_awkward()

# Particle stores masses in MeV; hepunits does the conversion for us.
MZ = Particle.from_pdgid(23).mass * MeV / GeV

# With XRootD, uproot streams just the branches it needs. Without it (e.g. on
# Windows), set USE_XROOTD = False: each file is then downloaded whole over
# HTTPS into a temporary directory that is removed when the script ends.
USE_XROOTD = True

if USE_XROOTD:
    BASE = "root://eospublic.cern.ch//eos/opendata/atlas/rucio/opendata"
else:
    BASE = "simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata"

# The two enabled by default run in well under a minute -- enough to check the pipeline.
# Uncomment the rest for the full 660 MB / 1.5M events, which is what actually gives the peak.
PERIODS = [
    "data16_periodL",
    "data16_periodI",
    # "data16_periodD",
    # "data16_periodG",
    # "data16_periodF",
    # "data16_periodC",
    # "data16_periodK",
    # "data15_periodJ",
    # "data16_periodB",
    # "data16_periodE",
    # "data15_periodG",
    # "data16_periodA",
    # "data15_periodE",
    # "data15_periodF",
    # "data15_periodH",
    # "data15_periodD",
]
FILES = [f"{BASE}/ODEO_FEB2025_v0_4lep_{p}.4lep.root" for p in PERIODS]

BRANCHES = [
    "lep_pt", "lep_eta", "lep_phi", "lep_e", 
    "lep_charge", "lep_type", "lep_isTightID", "lep_isTightIso",
]

# Chunk size for TTree.iterate.
STEP_SIZE = "25 MiB"

## The Analysis Logic
We need to group our branches into physics objects (4-momenta), apply quality cuts, build valid pairs for Z bosons, then build quads for the Higgs, and ultimately evaluate their invariant masses. This process involves complex combinatorics, which `awkward` handles beautifully.

In [ ]:
def four_lepton_mass(events):
    """Return the m_4l of the best 4-lepton candidate in each event."""
    
    # Group the branches into a single awkward array of Momentum4D records.
    leptons = ak.zip(
        {
            "pt": events.lep_pt,
            "eta": events.lep_eta,
            "phi": events.lep_phi,
            "E": events.lep_e,
            "charge": events.lep_charge,
            "flavour": events.lep_type,  # 11 = electron, 13 = muon
            "tight_id": events.lep_isTightID,
            "tight_iso": events.lep_isTightIso,
        },
        with_name="Momentum4D",
    )

    # 1. Object selection
    leptons = leptons[(leptons.pt > 5) & leptons.tight_id & leptons.tight_iso]
    leptons = leptons[
        (ak.num(leptons) == 4) & (ak.sum(leptons.charge, axis=1) == 0)
    ]

    # 2. Build Z candidates (pairs)
    pairs = ak.argcombinations(leptons, 2, fields=["i", "j"])
    pairs = pairs[
        (leptons[pairs.i].charge != leptons[pairs.j].charge) & 
        (leptons[pairs.i].flavour == leptons[pairs.j].flavour)
    ]

    # 3. Build Higgs candidates (quads) from pairs, ensuring distinct leptons
    quads = ak.combinations(pairs, 2, fields=["za", "zb"])
    za, zb = quads.za, quads.zb
    quads = quads[
        (za.i != zb.i) & (za.i != zb.j) & 
        (za.j != zb.i) & (za.j != zb.j)
    ]
    za, zb = quads.za, quads.zb

    # 4. Assign Z1 and Z2 with asymmetric mass windows
    mass_a = (leptons[za.i] + leptons[za.j]).mass
    mass_b = (leptons[zb.i] + leptons[zb.j]).mass
    a_is_leading = abs(mass_a - MZ) < abs(mass_b - MZ)
    
    m_z1 = ak.where(a_is_leading, mass_a, mass_b)
    m_z2 = ak.where(a_is_leading, mass_b, mass_a)
    
    keep = (m_z1 > 50) & (m_z1 < 106) & (m_z2 > 12) & (m_z2 < 115)
    quads, m_z1 = quads[keep], m_z1[keep]

    # 5. Select one best candidate per event
    best = ak.argmin(abs(m_z1 - MZ), axis=1, keepdims=True)
    quads = quads[best]
    
    za, zb = quads.za, quads.zb
    m4l = leptons[za.i] + leptons[za.j] + leptons[zb.i] + leptons[zb.j]
    
    return ak.drop_none(ak.flatten(m4l.mass))

## Processing the Data and Plotting
Now we iterate through our files in chunks using `uproot.iterate`, calculate the 4-lepton masses, and fill a histogram from `hist`. Finally, we plot our results!

In [ ]:
h = hist.Hist.new.Reg(29, 80, 250, name="m4l", label=r"$m_{4\ell}$ [GeV]").Double()

n_events = n_candidates = 0

for path in FILES:
    with uproot.open(path) as file:
        for chunk in file["analysis"].iterate(BRANCHES, step_size=STEP_SIZE):
            m4l = four_lepton_mass(chunk)
            h.fill(m4l)
            
            n_events += len(chunk)
            n_candidates += len(m4l)
            
print(f"events read: {n_events:,}")
print(f"total candidates: {n_candidates:,}")

for lo, hi, label in [
    (80, 100, "Z -> 4l"),
    (115, 130, "Higgs window"),
    (180, 250, "ZZ continuum"),
]:
    print(f" {label:14s} {h[hist.loc(lo):hist.loc(hi)].sum():4.0f}")

# Plotting
fig, ax = plt.subplots(figsize=(9, 5))
h.plot(ax=ax, histtype="fill", color="#5B8FF9", edgecolor="black", linewidth=0.8)

ax.axvline(125, color="crimson", linestyle="--", linewidth=1.2)
ax.text(127, ax.get_ylim()[1] * 0.75, r"$m_H = 125$ GeV", color="crimson")

ax.set_ylabel("Candidates / bin")
ax.set_title(r"ATLAS Open Data 2015+2016: $H \to ZZ^* \to 4\ell$")
plt.show()